In [3]:
import torch
from torch import nn
import torch.optim as optim

from torchvision import datasets, models, transforms

from torch.utils.data import DataLoader

import matplotlib.pyplot as plt

In [4]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

train_data = datasets.OxfordIIITPet(
    root="data",
    split="trainval",
    download=True,
    transform = transform
)

test_data = datasets.OxfordIIITPet(
    root="data",
    split="test",
    download=True,
    transform = transform
)

In [ ]:
from torchvision.models import resnet18
from torchvision.models import ResNet18_Weights

weights = ResNet18_Weights.DEFAULT

model = resnet18(weights=weights)
model.fc = torch.nn.Linear(512, 37)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Congelar toda la red
for parameter in model.parameters():
    parameter.requires_grad = False

# Descongelar únicamente la última capa
for parameter in model.fc.parameters():
    parameter.requires_grad = True
    
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32)

In [43]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

2.13.0+cpu
False


In [ ]:
def train(model, train_loader, criterion, optimizer, epochs=10, device="cpu"):
    model.train() # Cambia el modelo a modo entrenamiento
    model.to(device)
    
    loss_history = []
    accuracy_history = []
    
    for epoch in range(epochs):
        correct = 0 # Cantidad de imágenes clasificadas correctamente.
        total = 0 # Cantidad total de imágenes procesadas.
        total_loss = 0 # Suma del loss de todos los batches de la época.
        
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad() # Reinicia los gradientes acumulados del paso anterior.
            logits = model(images) # Forward: obtiene los logits del modelo.
            loss = criterion(logits, labels) # Calcula la pérdida comparando logits vs etiquetas reales.
            loss.backward() # Backpropagation: Calcula automáticamente todos los gradientes.
            optimizer.step() # Actualiza pesos de las convoluciones, de las capas densas y bias utilizando esos gradientes.
            
            total_loss += loss.item() # Acumula el loss de este batch.
            predictions = logits.argmax(dim=1) # Elige el logit más grande de cada imagen. Esa será la clase predicha.
            correct += (predictions == labels).sum().item() # Cuenta cuántas imágenes fueron clasificadas correctamente.
            total += labels.size(0) # Acumula la cantidad total de imágenes vistas.

        loss_history.append(total_loss / len(train_loader)) # Loss promedio considerando todos los batches.
        accuracy_history.append(correct / total) # Porcentaje de aciertos sobre todo el dataset.

        print(
            f"Epoch {epoch+1}/{epochs} "
            f"- Loss: {loss_history[-1]:.4f} "
            f"- Accuracy: {accuracy_history[-1]*100:.2f}%"
        )
    
    return loss_history, accuracy_history